# Описание проблемы

## Суть

В компании накоплен огромный объем критически важной технической информации – сотни Markdown-файлов с архитектурными решениями, инструкциями по устранению инцидентов и лучшими практиками.  
Однако доступ к этим знаниям практически отсутствует. Документация существует, но не работает – инженеры не могут быстро найти нужную информацию, особенно в стрессовых ситуациях.

Поиск ведется вручную: через `grep`, поиск по ключевым словам и постоянные вопросы старшим коллегам.  
Такой подход не масштабируется, тормозит развитие и снижает стабильность процессов.


## Сценарии

### 1. Концептуальный вопрос: *«Как это работает?»*

**Ситуация:**  
Новый инженер пытается понять, как устроен сервис авторизации — архитектура, базы данных, внешние зависимости.

**Действия:**
1. Ищет по слову `auth` в репозитории.  
2. Получает 30+ результатов — логи, старые задачи, API-примеры и устаревшие версии документации.  
3. Просматривает файлы вручную, пытаясь отличить актуальные данные от старых.  

**Последствия:**
- **Потеря времени:** задача занимает больше часа.  
- **Риск ошибки:** инженер может опереться на устаревшую информацию.  
- **Зависимость от людей:** в итоге он обращается к тимлиду, отвлекая его от основной работы.

### 2. Специфический вопрос: *«Что делать?»*

**Ситуация:**  
Ночью дежурный инженер получает алерт `High_CPU_Usage_on_Auth_Service`. Нужно срочно найти пошаговый плейбук.

**Действия:**
1. Ищет точное название алерта — безрезультатно: в документации он описан иначе.  
2. Пробует запросы “CPU”, “нагрузка”, “auth” — получает разрозненные файлы: архитектура, отчеты, старые инциденты.  
3. В критический момент тратит минуты на просмотр нерелевантных документов.

**Последствия:**
- **Увеличение MTTR:** каждая минута поиска — минута простоя сервиса.  
- **Риск ошибок:** использование устаревших инструкций может усугубить проблему.  


## Итог

Текущая система управления знаниями — один из главных технических долгов компании. Мы тратим тысячи часов на написание документации, но не получаем отдачи.  Знания рассыпаны, плохо индексируются и не помогают в работе.
Это не просто неудобство — это структурная проблема, замедляющая рост, повышающая риски и мешающая масштабировать команду.

# Основные подходы

## Продвинутый RAG (Advanced RAG)

Продвинутый RAG — это не единый алгоритм, а многоступенчатая архитектура, которая стала индустриальным стандартом для построения надёжных систем Retrieval-Augmented Generation.  
Её цель — повысить отношение «сигнал/шум» в контексте, подаваемом в LLM, снижая риск генерации неверных ответов и повышая точность извлечения.

Как систематизировано в обзоре **Gao et al. (2023), "Retrieval-Augmented Generation for Large Language Models: A Survey"** ([arXiv:2312.10997](https://arxiv.org/abs/2312.10997)), данный подход кодифицирует лучшие инженерные практики и устраняет ключевые недостатки наивных реализаций RAG.

### Механизм работы

#### 1. Гибридное извлечение (Hybrid Retrieval)

Этот компонент решает проблему разрыва между семантическим (векторным) и лексическим (BM25) поиском.  
Каждый из них эффективен в своей области, но ограничен при работе с реальными пользовательскими запросами:

* **Векторный поиск (Dense):** улавливает смысловые соответствия, но может пропустить точные совпадения терминов.  
* **Лексический поиск (Sparse, BM25):** точно находит совпадения, но не понимает синонимов и контекста.

Их объединение повышает устойчивость и полноту результатов.  
Эффективность гибридных методов подтверждена в рамках **BEIR** — обширного бенчмарка для оценки моделей информационного поиска (**Thakur et al., 2021**, *"BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models"*, [arXiv:2104.08663](https://arxiv.org/abs/2104.08663)).  
Для объединения результатов часто применяется алгоритм **Reciprocal Rank Fusion (RRF)**.  
Современные исследования, включая **Chen et al. (2024), "A Cheaper and Better Retrieval Method for RAG"** ([arXiv:2401.08092](https://arxiv.org/abs/2401.08092)), подтверждают, что гибридный подход остаётся одним из самых эффективных решений для работы с разнородными наборами данных.

#### 2. Двухэтапное извлечение с переранжированием (Retrieve & Re-rank)

Гибридный поиск обеспечивает широкий охват, но возвращает избыточное количество документов.  
Для фильтрации используется двухэтапная схема:

1. **Извлечение (Retrieve):** быстрый гибридный поиск формирует пул кандидатов (например, топ-50).  
2. **Переранжирование (Re-rank):** полученные документы оцениваются моделью **Cross-Encoder**, которая анализирует пары `(запрос, документ)` с использованием полного self-attention-механизма.

Преимущества Cross-Encoder-архитектуры продемонстрированы в работе **Nogueira & Cho (2019), "Passage Re-ranking with BERT"** ([arXiv:1901.04085](https://arxiv.org/abs/1901.04085)).  
Она позволяет улавливать более тонкие семантические связи и существенно повышает качество отбора релевантных документов.

### Преимущества

* **Высокая точность и надёжность:** комбинация гибридного поиска и переранжирования обеспечивает стабильные результаты на разных типах запросов.  
* **Снижение риска "галлюцинаций":** фильтрация контекста через Cross-Encoder уменьшает количество нерелевантных фрагментов, что напрямую снижает вероятность ошибок генерации.  
* **Хорошая адаптивность к масштабным корпусам:** метод стабильно работает при увеличении объёма корпоративных данных.

### Недостатки и ограничения

* **Повышенная задержка (latency):** Cross-Encoder требует значительных вычислительных ресурсов и увеличивает время отклика, что критично для интерактивных систем.   
* **Ограниченность контекста:** метод эффективно обрабатывает отдельные документы, но не строит связи между ними.  
  Он не способен ответить на вопросы, требующие анализа зависимостей между файлами, например: *«Какие сервисы затрагивает плейбук X?»* — видит релевантные документы, но не отношения между ними.

## Адаптивный RAG (Adaptive RAG) с самокоррекцией

Адаптивный RAG представляет собой переход от статичных конвейеров к динамическим системам, способным к **рефлексии и самокоррекции**.  
Если продвинутый RAG можно рассматривать как усовершенствованную сборочную линию, то адаптивный RAG — это система с встроенным контролем качества, которая способна остановить или скорректировать процесс при обнаружении ошибок.  

Ключевая идея — система не должна слепо доверять результатам поиска.  
Она анализирует промежуточные данные, принимает решения о корректировке стратегии и повторно запускает отдельные этапы при необходимости.  
Такой подход повышает надёжность и фактическую точность генерации ответов.

Источник: https://arxiv.org/abs/2401.15884

### Механизм работы

В отличие от линейной схемы продвинутого RAG, адаптивная архитектура использует **обратные связи и условные переходы**.  
Центральным элементом является **оценщик релевантности (relevance evaluator)**, который оценивает качество извлечённых документов до передачи их в LLM.

Эта концепция формализована в работе **Zhu et al. (2024), "Corrective Retrieval Augmented Generation (CRAG)"** ([arXiv:2401.15884](https://arxiv.org/abs/2401.15884)).  
Авторы описывают трёхуровневую классификацию извлечённых данных и соответствующие стратегии обработки:

1. **Корректное извлечение (высокая релевантность):**  
   Если документы полностью отвечают на запрос, они передаются в LLM без изменений.

2. **Неоднозначное извлечение (низкая релевантность):**  
   При частичном совпадении система выполняет **корректирующие действия** — например, **переформулирует запрос (Query Rewriting)**.  
   Такой подход описан в работе **Haq et al. (2023), "Query Rewriting for Retrieval-Augmented Large Language Models"** ([arXiv:2305.14283](https://arxiv.org/abs/2305.14283)).  
   LLM уточняет запрос и повторно запускает поиск.

3. **Некорректное извлечение (нулевая релевантность):**  
   Если документы нерелевантны, система полностью отбрасывает результаты и выполняет альтернативный, более широкий поиск.

Подобные механизмы самоконтроля и критической оценки также реализованы в архитектуре **Self-RAG**, предложенной **Asai et al. (2023), "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection"** ([arXiv:2310.11511](https://arxiv.org/abs/2310.11511)).  
Здесь модель самостоятельно оценивает качество контекста и своих ответов, что делает процесс генерации интерактивным и обучающимся в реальном времени.


### Преимущества

* **Высокая надёжность и снижение “галлюцинаций”:**  
  Встроенный механизм проверки контекста предотвращает генерацию ответов на основе нерелевантных данных.  
  Эксперименты **Zhu et al. (2024)** показывают значительное улучшение точности по сравнению со стандартными RAG-пайплайнами.

* **Оптимизация вычислительных ресурсов:**  
  Для простых запросов система использует короткий путь, а для сложных — применяет корректирующие циклы, достигая баланса между скоростью и качеством.

* **Устойчивость к нечетким запросам:**  
  Адаптивный RAG способен распознавать неоднозначные формулировки и самостоятельно их уточнять, избегая ложных ответов.

### Недостатки и ограничения

* **Усложнение архитектуры:**  
  Появляется дополнительный компонент — оценщик релевантности, требующий обучения, настройки и интеграции.  
  Это делает систему существенно сложнее в разработке и отладке.

* **Зависимость от точности оценщика:**  
  Ошибки модели-оценщика (например, признание нерелевантных данных корректными) могут приводить к новым, труднообнаружимым сбоям.

* **Непредсказуемое время отклика:**  
  Из-за циклов самокоррекции задержка ответа становится переменной.  
  Для простых запросов ответ может быть мгновенным, но для сложных — заметно увеличиваться, что ограничивает применимость в реальном времени.

## RAG на основе графов знаний (GraphRAG)

GraphRAG представляет собой переход от обработки документов как набора независимых текстовых фрагментов к представлению корпоративных знаний в виде **сети взаимосвязанных сущностей и отношений**.  
В отличие от классического RAG, ориентированного на семантическое сходство текстов, GraphRAG опирается на **явные логические связи** между объектами:  
вместо поиска «похожих документов» система отвечает на вопрос — *«как сущности в запросе связаны между собой в базе знаний?»*  

Такой подход особенно эффективен в доменах с чёткой структурой и зависимостями (например, `сервис` → `использует` → `базу данных`, `плейбук` → `решает` → `алерт`).  
Концепция синергии между LLM и графами знаний подробно рассмотрена в **Pan et al. (2023), "Unifying Large Language Models and Knowledge Graphs: A Roadmap"**  
([arXiv:2306.08302](https://arxiv.org/abs/2306.08302)).

Основной источник: https://arxiv.org/abs/2404.16130

### Механизм работы

GraphRAG реализуется в два этапа — **построение графа знаний (офлайн)** и **выполнение запросов (онлайн)**.

#### A. Построение графа знаний (Knowledge Graph Construction)


Этот этап является отдельной ML-задачей и включает в себя процесс **извлечения информации (Information Extraction, IE)**:

1. **Извлечение сущностей и отношений:**  
   Система анализирует корпус документов, выделяя ключевые сущности (*сервисы*, *алерты*, *плейбуки*) и устанавливая связи между ними в виде триплетов `(субъект, отношение, объект)` —  
   например, `(Auth_Service, USES, PostgreSQL)` или `(CPU_Playbook, RESOLVES, High_CPU_Alert)`.  
   Современные исследования, включая **Wadhwa et al. (2023), "Rethinking Knowledge Graph Construction from Text with LLMs"**  
   ([arXiv:2311.16226](https://arxiv.org/abs/2311.16226)), показывают, что крупные языковые модели способны эффективно выполнять эти задачи в режиме *zero-shot* и *few-shot*.

2. **Загрузка в графовую базу данных:**  
   Извлечённые триплеты загружаются в специализированную графовую СУБД (Neo4j, NebulaGraph и др.), формируя целостную структуру знаний.

#### B. Обработка запросов (Query Processing)

1. **Интерпретация запроса:**  
   Естественно-языковой запрос (например, *«Что делать при алерте `High_CPU_Usage`?»*) преобразуется в формальный графовый запрос (Cypher, SPARQL и т.п.).  
   Эта задача класса *Text-to-GraphQuery* решается с помощью LLM, выступающей в роли семантического переводчика.  
   Обзор методов представлен в **Guan et al. (2023), "A Survey on Graph-based Large Language Models"**  
   ([arXiv:2310.02982](https://arxiv.org/abs/2310.02982)).

2. **Поиск по графу (Graph Traversal):**  
   Выполняется запрос к графовой БД, который исследует не только узлы, но и пути между ними.  
   Например, система находит узел `Алерт: High_CPU_Usage` и следует по связи `RESOLVED_BY` к узлу `Плейбук: Scale_Pods.md`.

3. **Обогащение контекста и генерация ответа:**  
   Из графа извлекаются как структурированные данные (сущности и связи), так и связанный с ними неструктурированный контент (например, текст плейбука).  
   Этот контекст передаётся в LLM для синтеза точного и объяснимого ответа.  

   Практическая эффективность GraphRAG для *multi-document* задач подтверждается в работе  
   **Baek et al. (2023), "Knowledge Graph-based RAG for Multi-document Question Answering"**  
   ([arXiv:2311.13103](https://arxiv.org/abs/2311.13103)).

### Преимущества


* **Высокая точность и объяснимость:**  
  Система возвращает не только ответ, но и путь в графе, подтверждающий логику вывода.  

* **Решение сложных, многошаговых запросов:**  
  GraphRAG превосходит классические RAG-подходы при обработке *multi-hop* вопросов, где требуется объединить несколько источников данных.

* **Создание структурированного актива знаний:**  
  Граф знаний превращается в самостоятельный аналитический инструмент:  
  помогает выявлять пропуски (например, алерты без плейбуков) и визуализировать архитектуру всей системы.

### Недостатки и ограничения

* **Сложность и хрупкость построения графа:**  
  Качество системы зависит от точности извлечения сущностей и связей.  
  Даже современные LLM (см. **Wadhwa et al., 2023**) могут ошибаться, пропуская элементы или создавая ложные связи, что снижает достоверность графа.

* **Риск неполноты (Closed-World Assumption):**  
  Граф содержит только явно выявленные и корректно структурированные факты.  
  Неявные или двусмысленные отношения остаются вне модели, что ограничивает полноту ответов.

* **Слабая применимость к концептуальным вопросам:**  
  Для запросов вроде *«Какие лучшие практики по масштабированию?»* графовые методы неэффективны —  
  они требуют точных сущностей и связей. В таких случаях более результативен **векторный поиск**,  
  эффективность которого подтверждена в бенчмарке **BEIR (Thakur et al., 2021)**  
  ([arXiv:2104.08663](https://arxiv.org/abs/2104.08663)).

# MVP

| Компонент | Технология | Назначение |
|-----------|------------|------------|
| **Vector DB** | Milvus | Семантический поиск по эмбеддингам |
| **Full-text Search** | OpenSearch | Полнотекстовый поиск с n-gram |
| **Embedding Model** | FRIDA (ai-forever) | Русскоязычные эмбеддинги |
| **Reranker** | BGE-Reranker-v2-m3 | Cross-Encoder для реранкинга |
| **LLM** | qwen2.5-7b-instruct | Генерация ответов |
| **API Framework** | FastAPI | HTTP API |
| **Configuration** | Pydantic Settings | Типизированная конфигурация |
| **Observability** | OpenTelemetry | Трейсинг |

## Арха

![Архитектура](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/RAG_ITMO.drawio.png)

## Hybrid Search + RRF Fusion

![Поиск](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/Hybrid_search.jpg)

## Parent-Document Retrieval

![PDR](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/Parent_document.png)

Ищем по маленьким чанкам (точность), возвращаем полный раздел (контекст).

## Two-Stage Reranking

![Reranker](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/Reranker.jpg)

![LLM-Reranker](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/LLM_reranking.jpg)

LLM как второй реранкер.
Не только оценивает релевантность, но и создаёт summary для контекст-сжатия.

## Query Expansion

![Расширение запроса](/Users/iamnoob/Study/ITMO/RAG/research/Notebooks/pics/Query.jpg)

```python
# Пример
"нагрузка на auth" → [
    "High CPU Usage Auth Service",
    "высокая нагрузка сервис авторизации",
    "auth service performance"
]
```

## Context Compression via Structured Output

LLM фильтрует и сжимает контекст ДО финальной генерации → меньше токенов, выше качество.

## AST-Parsing для Markdown

Блоки кода и таблицы извлекаются через AST

TODO: ПОМЕНЯТЬ НА ТУ ЧТО ЛУЧШЕ ДРУЖИТ С ГИТХАБОВСКИМ ВАРИНАТОМ. GFM ВРОДЕ

## TODO: SGR (?)

## TODO: AUTO TESTING (ПЕРЕДЕЛАТЬ НА RAGAS)

Сделать намного больше синтетики. Либо ручками, либо через спец. фреймворки сделать золотой датасет вопросов с ответами, прокинуть квен побольше через API, гонять тесты и замерять качество.

Метрики:
- Context Recall (Весь ли мы релевантный контекст используем)
- Context Precision@k (Много ли шума в переданном контексте)
- Precision@k 
- Faithfulness Score (Насколько модель использует переданный контекст)

## OPTIMIZATION PROMPTS + BETTER CONFIGS

Просто лучше промпты сделать, сейчас они за 5 минут написаны были (КОД НЕ ПЕРЕФРАЗ)

все параметры тоже получше подобрать, так как от балды поставлены

## BETTER LLM

Базовая: qwen32b. 

Судья: Qwen3 235B A22B Thinking 2507

# TODO: MVP+

Добавить vLLM для описание картинок + добавить в выдачу

# TODO: MVP++, AGENTIC RAG

Пусть он ищет делает вызов инструментов, может вызывать код, ходить в интернет и так далее